In [8]:
from notebookutils import mssparkutils
from pyspark.sql import functions as F

account   = "synpasetofabric"
container = "fabmigation1"

bronze_base = f"abfss://{container}@{account}.dfs.core.windows.net/bronze/adventureworks2/"
silver_base = f"abfss://{container}@{account}.dfs.core.windows.net/silver/adventureworks/"

print("BRONZE:", bronze_base)
print("SILVER:", silver_base)

In [9]:
files = mssparkutils.fs.ls(bronze_base)

purchasing_files = [
    f"{bronze_base}{f.name}"
    for f in files
    if f.name.startswith("Purchasing.") and f.name.endswith(".parquet")
]


print("Purchasing.*  files found:")
for p in purchasing_files:
    print(" -", p)

In [10]:
def clean_df(df):
    # trim() a strings
    for col_name, dtype in df.dtypes:
        if dtype == "string":
            df = df.withColumn(col_name, F.trim(F.col(col_name)))

    # Replace nulls
    fill_dict = {col: "" for col, dtype in df.dtypes if dtype == "string"}
    fill_dict_numeric = {col: 0 for col, dtype in df.dtypes if dtype in ["int", "bigint", "double"]}
    df = df.fillna(fill_dict).fillna(fill_dict_numeric)

    return df

In [11]:
from typing import List, Optional
from pyspark.sql import DataFrame, functions as F, types as T

def standardizedate_df(
    df: DataFrame,
    date_cols: Optional[List[str]] = None,
    input_formats: Optional[List[str]] = None,
    cast_dates_as: str = "date",     # "date" or "timestamp"
    ingest_date: Optional[str] = None  # e.g., "2026-02-03" (casts to date); if None -> current_date()
) -> DataFrame:
    """
    1) Standardizes the provided or auto-detected date/timestamp columns:
       - Parses common string formats to timestamp
       - Casts to `date` (default) or keeps as `timestamp` via `cast_dates_as`
    2) Adds `ingest_date` column (date) for downstream partitioning.

    Parameters
    ----------
    df : input DataFrame
    date_cols : list of column names to treat as dates/timestamps. If None, auto-detects by name/ctype.
    input_formats : list of strptime-style formats to try in order
    cast_dates_as : "date" or "timestamp"
    ingest_date : string literal yyyy-MM-dd; if None uses current_date()
    """

    # --- 0) Trim strings
    for col_name, dtype in df.dtypes:
        if dtype == "string":
            df = df.withColumn(col_name, F.trim(F.col(col_name)))

    # --- 1) Which columns are date-like?
    if date_cols is None:
        # Heuristic: names that look like dates & are string/date/timestamp
        keywords = ("date", "dt", "_at", "_on", "timestamp", "ts", "fecha", "created", "updated")
        date_cols = [
            c for c, t in df.dtypes
            if t in ("string", "date", "timestamp") and any(k in c.lower() for k in keywords)
        ]

    # --- 2) Parse formats for string dates
    if input_formats is None:
        input_formats = [
            # Date only
            "yyyy-MM-dd", "yyyy/MM/dd", "dd-MM-yyyy", "dd/MM/yyyy", "MM-dd-yyyy", "MM/dd/yyyy",
            "yyyyMMdd", "ddMMyyyy",
            # Date + time
            "yyyy-MM-dd HH:mm:ss", "dd/MM/yyyy HH:mm:ss", "MM/dd/yyyy HH:mm:ss",
            # ISO-ish / with millis / TZ
            "yyyy-MM-dd'T'HH:mm:ss", "yyyy-MM-dd'T'HH:mm:ss.SSS",
            "yyyy-MM-dd'T'HH:mm:ssXXX", "yyyy-MM-dd'T'HH:mm:ss.SSSXXX"
        ]

    dtype_map = dict(df.dtypes)
    for c in date_cols:
        t = dtype_map.get(c, "")

        # If it's already timestamp/date, just unify type later
        if t == "string":
            # Try multiple formats with coalesce
            parsed = None
            for fmt in input_formats:
                candidate = F.to_timestamp(F.col(c), fmt)
                parsed = candidate if parsed is None else F.coalesce(parsed, candidate)

            # Epoch seconds / millis (string or numeric) fallbacks
            parsed = F.coalesce(
                parsed,
                F.to_timestamp(F.col(c).cast("double")),                    # epoch seconds
                F.to_timestamp((F.col(c).cast("double")/1000.0))            # epoch millis
            )

            # If nothing matched, keep original to avoid data loss
            df = df.withColumn(c, F.when(parsed.isNotNull(), parsed).otherwise(F.col(c)))

        # Finally, cast to the requested final type
        if cast_dates_as == "date":
            df = df.withColumn(c, F.to_date(F.col(c)))
        else:
            df = df.withColumn(c, F.to_timestamp(F.col(c)))

    # --- 3) Add ingest_date for partitioning
    if ingest_date:
        df = df.withColumn("ingest_date", F.lit(ingest_date).cast(T.DateType()))
    else:
        df = df.withColumn("ingest_date", F.current_date())

    return df

In [12]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import col, when

def add_unit_price_validation_flag(
    df: DataFrame,
    column_name: str = "UnitPrice",
    flag_column: str = "is_valid_unit_price"
) -> DataFrame:
    """
    Agrega una columna booleana indicando si UnitPrice es válido (>= 0).
    """
    return df.withColumn(
        flag_column,
        when(col(column_name).isNotNull() & (col(column_name) >= 0), True)
        .otherwise(False)
    )

In [13]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import col, when, lit

def add_shipdate_ge_orderdate_flag(
    df: DataFrame,
    ship_col: str = "ShipDate",
    order_col: str = "OrderDate",
    flag_column: str = "is_valid_shipdate"
) -> DataFrame:
    """
    Agrega un flag booleano:
    - True si ShipDate >= OrderDate
    - True si ShipDate es NULL
    - False si ShipDate < OrderDate
    """
    return df.withColumn(
        flag_column,
        when(col(ship_col).isNull(), lit(True))
        .when(col(ship_col) >= col(order_col), lit(True))
        .otherwise(lit(False))
    )

In [14]:
for file_path in purchasing_files:
    print(f"\nProcessing: {file_path}")

    # Base name, ej: Production.Culture.parquet
    file_name = file_path.split("/")[-1].replace(".parquet", "")

    silver_table_name = (
        "silver_" +
        file_name
            .replace(".", "_")
            .lower()
    )

    # Read parquet file
    df = spark.read.parquet(file_path) 
    df.printSchema()

    # Clean
    df_clean = clean_df(df)

    # dates
    df_table= standardizedate_df(df_clean)
    
    # ✅ UnitPrice validation flag (solo si existe la columna)
    if "UnitPrice" in df_table.columns:
            df_table = add_unit_price_validation_flag(
                df_table,
                column_name="UnitPrice",
                flag_column="is_valid_unit_price"
            )
            print(f"✅ Added is_valid_unit_price for {silver_table_name}")
    else:
            print(f"⏭️ {silver_table_name} no tiene columna UnitPrice, se omite validación")

    # ShipDate vs OrderDate validation (solo si existen ambas columnas)
    if {"ShipDate", "OrderDate"}.issubset(df_table.columns):
        df_table = add_shipdate_ge_orderdate_flag(
            df_table,
            ship_col="ShipDate",
            order_col="OrderDate",
            flag_column="is_valid_shipdate"
        )
        print(f"✅ Added is_valid_shipdate for {silver_table_name}")
    else:
        print(f"⏭️ {silver_table_name} no tiene ShipDate y OrderDate, se omite validación")

    # Destination silver path
    out_path = f"{silver_base}{silver_table_name}/"

    print(f"Saved in Silver: {out_path}")

    # Save as parquet (Silver)
    
    (
        df_table
        .write
        .mode("overwrite")
        .partitionBy("ingest_date")
        .parquet(out_path)
    )
    